# 2026 World Cup ETL primary process

This first exercise will perform a first extraction to API-Football, ideally we are going to consider the 48 teams that will participate in the world cup and fetch their results on the last 2 years (the time window will be expanded if possible and necessary).

First of all we will try to connect to the API, and get the codes for the national teams

In [9]:
#%pip install -r requirements.txt

In [12]:
import os
import json
import time
import random
from datetime import datetime
from dotenv import load_dotenv
from libs.api_client import APIFootballClient

# ============================
# CONFIGURACIÓN GENERAL
# ============================

START_YEAR = 2022
END_YEAR = 2024
REQUEST_LIMIT = 100

# Reinicio manual por grupo
START_GROUP = 0   # ← Cambia este valor para reiniciar donde quieras

request_count = 0
team_id_cache = {}
problematic_cases = []

load_dotenv()

OUTPUT_DIR = "input"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Rango de años a extraer (ajustable)
START_YEAR = 2022
END_YEAR = 2024

TEAMS = [
    # Grupo A
    "Mexico", "South Korea", "South Africa", "Czech Republic",
    # Grupo B
    "Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland",
    # Grupo C
    "Brazil", "Morocco", "Haiti", "Scotland",
    # Grupo D
    "USA", "Australia", "Paraguay", "Turkey",
    # Grupo E
    "Germany", "Ecuador", "Ivory Coast", "Curacao",
    # Grupo F
    "Netherlands", "Japan", "Sweden", "Tunisia",
    # Grupo G
    "Belgium", "Iran", "Egypt", "New Zealand",
    # Grupo H
    "Spain", "Uruguay", "Saudi Arabia", "Cape Verde",
    # Grupo I
    "France", "Senegal", "Iraq", "Norway",
    # Grupo J
    "Argentina", "Algeria", "Austria", "Jordan",
    # Grupo K
    "Portugal", "Colombia", "Uzbekistan", "Congo DR",
    # Grupo L
    "England", "Croatia", "Ghana", "Panama"
]

In [13]:
# ============================================================
# 1. GROUP GENERATOR
# ============================================================

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

TEAM_GROUPS = list(chunk_list(TEAMS, 4))

client = APIFootballClient()
team_id_cache = {}

In [14]:
# ============================================================
# 2. CONTADOR DE REQUESTS
# ============================================================

def counted_request(func, *args, **kwargs):
    global request_count

    if request_count >= REQUEST_LIMIT:
        print(f"\n🛑 Límite diario alcanzado ({REQUEST_LIMIT} requests). Deteniendo extracción.\n")
        return None

    result = func(*args, **kwargs)
    request_count += 1

    print(f"   🔢 Request #{request_count}/{REQUEST_LIMIT}")

    return result

In [15]:
# ============================================================
# 3. DASHBOARD
# ============================================================

def print_dashboard(group_idx, total_groups, team_idx, group, team_name, year):
    print("\n--------------------------------------------------")
    print(f"📊 Progreso")
    print(f"   Grupo: {group_idx+1}/{total_groups}")
    print(f"   Equipo en grupo: {team_idx+1}/{len(group)} — {team_name}")
    print(f"   Año: {year}")
    print(f"   Requests usados: {request_count}/{REQUEST_LIMIT}")
    print(f"   Requests restantes: {max(0, REQUEST_LIMIT - request_count)}")
    print("--------------------------------------------------\n")

In [16]:

# ============================================================
# 4. FUNCIONES ROBUSTAS
# ============================================================

def get_team_id_cached(team_name, retries=3):
    if team_name in team_id_cache:
        return team_id_cache[team_name]

    for i in range(retries):
        team_id = counted_request(client.get_national_team_id, team_name)

        if team_id and team_id != 0:
            team_id_cache[team_name] = team_id
            return team_id

        sleep_time = 2 ** i
        print(f"⚠️ ID inválido para {team_name}. Retry {i+1}/{retries} — {sleep_time}s...")
        time.sleep(sleep_time)

    print(f"❌ No se pudo obtener ID para {team_name} después de {retries} intentos.")
    problematic_cases.append({
        "team": team_name,
        "issue": "team_id_not_found_after_retries"
    })

    team_id_cache[team_name] = None
    return None

def retry_team_id(team_name, retries=3):
    for i in range(retries):
        sleep_time = 2 ** i
        print(f"   ↳ Retry ID {i+1}/{retries} — {sleep_time}s...")
        time.sleep(sleep_time)

        team_id = counted_request(client.get_national_team_id, team_name)
        if team_id and team_id != 0:
            print(f"   ✔ ID recuperado: {team_id}")
            return team_id

    print(f"   ❌ No se pudo obtener ID válido para {team_name}")
    return None


def safe_get_fixtures(team_id, year, retries=3):
    if team_id is None:
        return []

    for i in range(retries):
        data = counted_request(client.get, "fixtures", {"team": team_id, "season": year})

        if data and "response" in data and len(data["response"]) > 0:
            return data["response"]

        sleep_time = 2 ** i
        print(f"   ↳ Fixtures vacíos. Retry {i+1}/{retries} — {sleep_time}s...")
        time.sleep(sleep_time)

    print(f"   ❌ No se pudieron obtener fixtures para {team_id} en {year}")
    problematic_cases.append({
        "team_id": team_id,
        "year": year,
        "issue": "fixtures_empty_after_retries"
    })

    return []

In [17]:
# ============================================================
# 5. EXTRACCIÓN PRINCIPAL
# ============================================================

def extract_group(group_idx, group, total_groups):
    all_fixtures = []

    print(f"\n==============================")
    print(f"   Extrayendo grupo {group_idx+1}/{total_groups}: {group}")
    print(f"==============================\n")

    for team_idx, team in enumerate(group):
        if request_count >= REQUEST_LIMIT:
            break

        print(f"➡️ Equipo: {team}")

        team_id = get_team_id_cached(team)

        if team_id is None:
            print(f"   ⚠️ Saltando {team} (ID no encontrado).")
            continue

        print(f"   ✔ ID encontrado: {team_id}")

        for year in range(START_YEAR, END_YEAR + 1):
            if request_count >= REQUEST_LIMIT:
                break

            print_dashboard(group_idx, total_groups, team_idx, group, team, year)

            fixtures = safe_get_fixtures(team_id, year)
            all_fixtures.extend(fixtures)

            # Cooldown por equipo
            time.sleep(random.uniform(1.5, 3.0))

    # STOP si no hubo fixtures válidos
    if not all_fixtures:
        print("\n🛑 No se obtuvieron fixtures válidos en este grupo. Deteniendo extracción.\n")
        return all_fixtures, True

    # Cooldown por grupo
    print("😴 Cooldown de 90 segundos antes del siguiente grupo...")
    time.sleep(90)

    return all_fixtures, False

In [18]:
# ============================================================
# 6. GUARDADO
# ============================================================

def save_group_fixtures(group_index, fixtures):
    if not fixtures:
        print(f"\nℹ️ Grupo {group_index+1}: sin fixtures para guardar.\n")
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{OUTPUT_DIR}/fixtures_group_{group_index+1}_{timestamp}.json"

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(fixtures, f, indent=2)

    print(f"\n💾 Guardado: {filename}\n")

In [20]:
print("🚀 Iniciando extracción local...\n")
total_groups = len(TEAM_GROUPS)
all_results = []

for group_idx in range(START_GROUP, len(TEAM_GROUPS)):
    fixtures, stop = extract_group(group_idx, TEAM_GROUPS[group_idx], len(TEAM_GROUPS))
    all_results.extend(fixtures)

    if stop:
        break
    
print("\n🎉 Extracción completada (o detenida por límite/ID).")
print(f"   Requests usados: {request_count}/{REQUEST_LIMIT}")
print("   Revisa los archivos en la carpeta raw_fixtures.\n")

🚀 Iniciando extracción local...


   Extrayendo grupo 1/12: ['Mexico', 'South Korea', 'South Africa', 'Czech Republic']

➡️ Equipo: Mexico
   🔢 Request #1/100
   ✔ ID encontrado: 16

--------------------------------------------------
📊 Progreso
   Grupo: 1/12
   Equipo en grupo: 1/4 — Mexico
   Año: 2022
   Requests usados: 1/100
   Requests restantes: 99
--------------------------------------------------

   🔢 Request #2/100

--------------------------------------------------
📊 Progreso
   Grupo: 1/12
   Equipo en grupo: 1/4 — Mexico
   Año: 2023
   Requests usados: 2/100
   Requests restantes: 98
--------------------------------------------------

   🔢 Request #3/100

--------------------------------------------------
📊 Progreso
   Grupo: 1/12
   Equipo en grupo: 1/4 — Mexico
   Año: 2024
   Requests usados: 3/100
   Requests restantes: 97
--------------------------------------------------

   🔢 Request #4/100
➡️ Equipo: South Korea
   🔢 Request #5/100
   ✔ ID encontrado: 17

-------

In [21]:
with open("problematic_cases.json", "w") as f:
    json.dump(problematic_cases, f, indent=2)